## 🧩 第一步：认识 Tensor —— PyTorch 的“灵魂”

在 PyTorch 里，几乎所有东西（数据、参数、输入、输出）都是 Tensor（张量）

👉 它是 多维数组，类似于 NumPy 的 ndarray，但可以在 GPU 上加速运算。


In [ ]:
import torch

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("x=\n", x)

print("shape:", x.shape)
print("dtype:", x.dtype)
print("device", x.device)

a = torch.zeros((2, 3))
b = torch.ones((2, 3))
c = torch.randn((2, 3))
print("a = ", a)
print("b = ", b)
print("c = ", c)

x=
 tensor([[1., 2.],
        [3., 4.]])
shape: torch.Size([2, 2])
dtype: torch.float32
device cpu
a =  tensor([[0., 0., 0.],
        [0., 0., 0.]])
b =  tensor([[1., 1., 1.],
        [1., 1., 1.]])
c =  tensor([[ 1.5550, -0.5703, -1.3767],
        [ 0.8777,  1.0235, -0.6726]])


## 🧩 第二步：张量的基本运算（Tensor Operations）

我们主要学习以下几个方面：

1. 张量的形状操作（如 reshape、view、unsqueeze、squeeze）

2. 张量的数学运算（加减乘除、矩阵乘法）

3. 广播机制（Broadcasting）

4. 索引与切片

### 1. 张量的形状操作（如 reshape、view、unsqueeze、squeeze）

#### 🧠一、reshape 与 view 的区别


| 函数          | 功能               | 底层机制 | 是否创建新内存    |
| ----------- | ---------------- | ---- | ---------- |
| `reshape()` | 改变形状，自动选择是否创建新内存 | 智能   | 有时创建新张量    |
| `view()`    | 改变形状，但要求内存连续     | 不智能  | 必须保证张量是连续的 |


个人总结：
在 PyTorch（乃至 NumPy）中，张量的元素通常在内存中是连续排列的。
但某些操作（比如 .t() 转置、切片）会改变访问顺序，使它不再连续。

view(): 

要求张量必须是连续的。
因为 view() 只是重新解释内存布局，它不会复制数据。
如果张量不是连续的，它就无法直接“重新解释”，因此会报错。

reshape() 会尝试：
1. 如果可能，不复制数据（直接用 view）；
2. 如果不行，就自动复制一份新的连续内存。

一般而言就是转置过的张量无法view，不追求性能就用万能的reshape


#### 🧠二、unsqueeze 和 squeeze


##### 🧪unsqueeze

x = torch.tensor([1, 2, 3])  # shape: (3,)

那么只能

x.unsqueeze(0)  # 在第0个维度插入 → shape: (1, 3)

x.unsqueeze(1)  # 在第1个维度插入 → shape: (3, 1)

看shape就好，参数为几就表示在shape下标为几的地方加一个维度

In [25]:
x = torch.tensor([1, 2, 3])
print(x.shape)
x.unsqueeze(0)
print(x.shape)

torch.Size([3])
torch.Size([3])


为什么还没变呢？

因为unsqueeze函数不会在原地修改，而是将修改结果作为返回值返回

下面这样就好了

In [26]:
x = torch.tensor([1, 2, 3])
print(x.shape)
x = x.unsqueeze(0)
print(x.shape)

torch.Size([3])
torch.Size([1, 3])


✅ PyTorch 约定俗成：
带下划线 _ 结尾的函数（如 .add_()、 .zero_()、 .unsqueeze_()）表示“原地修改”。

##### 🧪squeeze

squeeze(dim)

删除大小为 1 的维度。

In [29]:
x = torch.randn(1, 3, 1, 5)
x.shape   # torch.Size([1, 3, 1, 5])

print(x.squeeze().shape)        # torch.Size([3, 5]) → 删除所有为1的维度
print(x.squeeze(0).shape)       # torch.Size([3, 1, 5]) → 仅删除第0个维度
print(x.squeeze(1).shape)       # torch.Size([1, 3, 1, 5]) 大小不为1的维度删不掉！！！

torch.Size([3, 5])
torch.Size([3, 1, 5])
torch.Size([1, 3, 1, 5])


🌰 举个实际例子

在神经网络中，模型输入通常需要形状 [batch_size, channel, height, width]。
如果你只有一张灰度图（没有 batch 维度），就可以：

In [32]:
img = torch.randn(1, 28, 28)
img = img.unsqueeze(0)  # shape: (1, 1, 28, 28)
img.shape


torch.Size([1, 1, 28, 28])

### 2. 张量的数学运算（加减乘除、矩阵乘法）

PyTorch 张量的加减乘除有三种常见写法，效果等价：

In [1]:
import torch

a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

print(a + b)
print(torch.add(a, b))
print(a.add(b))

tensor([5, 7, 9])
tensor([5, 7, 9])
tensor([5, 7, 9])


矩阵乘法有两种写法
torch.matmul 和@

In [2]:
x = torch.tensor([[1, 2], [3, 4]])
w = torch.tensor([[5, 6], [7, 8]])

print(torch.matmul(x, w))
print(x @ w)

tensor([[19, 22],
        [43, 50]])
tensor([[19, 22],
        [43, 50]])


### 3. 广播机制（Broadcasting）

广播是 PyTorch 的一个强大特性：
当两个张量形状不一致时，只要满足一定规则，也可以运算。

In [4]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
y = torch.tensor([10, 20, 30])

print(x + y)

tensor([[11, 22, 33],
        [14, 25, 36]])


练习

In [5]:
import torch

A = torch.randn(2, 3)
B = torch.randn(3)
C = torch.randn(1, 3)
D = torch.randn(2, 1)

print((A + B).shape)  # (2, 3)
print((A + C).shape)  # (2, 3)
print((A + D).shape)  # (2, 3)

torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])


### 4. 索引与切片

In [6]:
import torch

x = torch.tensor([[10, 11,12], 
                  [13, 14, 15], 
                  [16, 17, 18]])

print(x)

tensor([[10, 11, 12],
        [13, 14, 15],
        [16, 17, 18]])


1. 访问单个元素

In [8]:
print(x[0, 0])
print(x[1, 2])

tensor(10)
tensor(15)


2. 访问整行或整列

In [9]:
print(x[0])
print(x[:, 1])

tensor([10, 11, 12])
tensor([11, 14, 17])


3. 切片操作(slicing)

In [10]:
print(x[0:2, 1:3])

tensor([[11, 12],
        [14, 15]])


3. 负索引从末尾开始计数

In [11]:
print(x[-1])
print(x[:, -1])

tensor([16, 17, 18])
tensor([12, 15, 18])


4. 步长（step）允许隔行或隔列取值

In [12]:
print(x[::2, ::2])

tensor([[10, 12],
        [16, 18]])


In [14]:
print(x[1::1, 1::1])

tensor([[14, 15],
        [17, 18]])


5. 布尔索引

In [16]:
mask = x > 15
print(mask)
print(x[mask])

tensor([[False, False, False],
        [False, False, False],
        [ True,  True,  True]])
tensor([16, 17, 18])


6. 高级索引

In [17]:
idx = torch.tensor([0, 2])
print(torch.index_select(x, 0, idx))


tensor([[10, 11, 12],
        [16, 17, 18]])


练习

In [18]:
import torch

x = torch.tensor([[ 1,  2,  3,  4],
                  [ 5,  6,  7,  8],
                  [ 9, 10, 11, 12]])
print(x[1:, 2:])        # [[7, 8], [11, 12]]
print(x[:2, ::2])       # [[1, 3], [5, 7], [9, 11]]
mask = x % 2 == 0
print(x[mask])          # [[2, 4], [6, 8], [10, 12]]

tensor([[ 7,  8],
        [11, 12]])
tensor([[1, 3],
        [5, 7]])
tensor([ 2,  4,  6,  8, 10, 12])


## 🧩 第三步：自动求导与计算图（Autograd）

在 PyTorch 中，requires_grad=True 的张量会被追踪计算过程，自动构建计算图。
然后可以调用 .backward() 来计算梯度。

In [21]:
import torch

x = torch.tensor([2.0], requires_grad=True)
y = x ** 2 + 3 * x + 1
print(y)

y.backward()

print("x.grad:", x.grad)

tensor([11.], grad_fn=<AddBackward0>)
x.grad: tensor([7.])


梯度累加

In [25]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()
print(x.grad)

y = (x ** 3).sum()
y.backward()
print(x.grad)

tensor([2., 4., 6.])
tensor([ 5., 16., 33.])


## 🧩 第四步：线性回归与反向传播实战

In [30]:
import torch

x = torch.tensor([[1.0], [2.0], [3.0]])
y_true = torch.tensor([[2.0], [4.0], [6.0]])

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.1

for epoch in range(50):
    y_pred = w * x + b
    
    loss = ((y_pred - y_true) ** 2).mean()
    
    loss.backward()
    
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    
    w.grad.zero_()
    b.grad.zero_()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | Loss: {loss.item():.6f} | w: {w.item():.3f}, b: {b.item():.3f}")
        
print(f"\n最终模型: y = {w.item():.3f}x + {b.item():.3f}")

Epoch  0 | Loss: 45.827435 | w: 2.007, b: 0.764
Epoch 10 | Loss: 0.037635 | w: 1.780, b: 0.500
Epoch 20 | Loss: 0.023133 | w: 1.828, b: 0.392
Epoch 30 | Loss: 0.014220 | w: 1.865, b: 0.307
Epoch 40 | Loss: 0.008740 | w: 1.894, b: 0.241

最终模型: y = 1.915x + 0.194
